# Smart Agriculture Monitoring — Irrigation Prediction (Google Colab)

**Aim:** Predict whether a crop needs irrigation, from sensor readings.

**Run note:** In this copy the dataset is created in the first cell (same data, same fixed seed as `agriculture_sensor_data.csv`), so the notebook runs end to end with Runtime → Run all and no file upload step. The investigation cells are identical to the upload version.

In [ ]:
# Create the dataset in this notebook (same data as agriculture_sensor_data.csv,
# same fixed seed, so the results match the CSV version exactly)
import numpy as np
import pandas as pd

CROPS_MIN_MOISTURE = {"wheat": 40, "rice": 60, "tomato": 50, "onion": 45}
CROPS = list(CROPS_MIN_MOISTURE)

np.random.seed(42)
n = 300
crop = np.random.choice(CROPS, size=n)
soil_moisture = np.round(np.random.uniform(20, 90, size=n), 1)
temperature = np.round(np.random.uniform(15, 40, size=n), 1)
humidity = np.round(np.random.uniform(40, 95, size=n), 1)
light = np.random.randint(100, 1100, size=n)
irrigation_needed = [1 if m < CROPS_MIN_MOISTURE[c] else 0
                     for m, c in zip(soil_moisture, crop)]

df = pd.DataFrame({
    "crop": crop,
    "soil_moisture": soil_moisture,
    "temperature": temperature,
    "humidity": humidity,
    "light": light,
    "irrigation_needed": irrigation_needed,
})
print("Dataset shape:", df.shape)
print(df.head())

## Investigation 1: Understanding Dataset

In [ ]:
# Investigation 1: Understanding Dataset
print(df.columns)
print("\nTarget Variable: irrigation_needed (1 = irrigate, 0 = no need)")
print(df["irrigation_needed"].value_counts())
print("\nFeatures:", [c for c in df.columns if c != "irrigation_needed"])
print("\nDataset shape:", df.shape)
print(df.head())

## Investigation 2: Preparing Data for Logistic Regression

In [ ]:
# Investigation 2: Preparing Data for Logistic Regression
from sklearn.model_selection import train_test_split

# One-Hot Encode the crop names (first crop kept as baseline)
df_enc = pd.get_dummies(df, columns=["crop"], drop_first=True)
X = df_enc.drop("irrigation_needed", axis=1)
y = df_enc["irrigation_needed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))
print("Number of Features:", X.shape[1])
print(X_train.head())

## Investigation 3: Building and Training the Logistic Regression Model

In [ ]:
# Investigation 3: Building and Training the Logistic Regression Model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Intercept:", model.intercept_)
print("\nRegression Coefficients:")
for feature, coef in zip(X.columns, model.coef_[0]):
    print(f"{feature} : {coef}")

## Investigation 4: Predicting Irrigation Need using the Logistic Regression Model

In [ ]:
# Investigation 4: Predicting Irrigation Need using the Logistic Regression Model
y_pred = model.predict(X_test)

results = pd.DataFrame(
    {"Actual Status": y_test.values, "Predicted Status": y_pred})
print(results.head(10))

# Predict for one new reading (built as a DataFrame, so feature names stay valid)
new_raw = pd.DataFrame([{
    "crop": "wheat", "soil_moisture": 28, "temperature": 34,
    "humidity": 55, "light": 950,
}])
new_X = pd.get_dummies(new_raw, columns=["crop"], drop_first=True)
new_X = new_X.reindex(columns=X.columns, fill_value=0)
print("\nPredicted Status:",
      ["YES - Irrigate now" if p == 1 else "NO - Soil is good"
       for p in model.predict(new_X)])

results["Correct Prediction"] = (
    results["Actual Status"] == results["Predicted Status"])
print(results.head(10))

## Investigation 5: Evaluating the Performance of the Logistic Regression Model

In [ ]:
# Investigation 5: Evaluating the Performance of the Logistic Regression Model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

## Investigation 6: Interpreting the Logistic Regression Model for Decision-Making

In [ ]:
# Investigation 6: Interpreting the Logistic Regression Model for Decision-Making
coef_df = pd.DataFrame(
    {"Feature": X.columns, "Coefficient": model.coef_[0]})
print(coef_df)
print("Intercept:", model.intercept_[0])
ranked = coef_df.assign(
    Absolute_Coefficient=coef_df["Coefficient"].abs()
).sort_values("Absolute_Coefficient", ascending=False)
print(ranked)

## Save the trained model

In [ ]:
# Save the trained model to a .pkl file
import pickle

with open("agriculture_model.pkl", "wb") as file:
    pickle.dump(model, file)
print("Model saved as agriculture_model.pkl")